# Comparing stocks

One table across every stock run with `python run.py`, built from the summaries each report saved in `reports/summaries/`.

**The lucky-winners problem.** Test enough stocks and models, and some will "win" by chance alone. At the usual 5% level, about 1 comparison in 20 passes even when nothing real is going on. So this report shows two bars:

- **5%:** the usual threshold for a single test.
- **Corrected:** 5% divided by the number of comparisons (the *Bonferroni correction*). A result has to clear this to count as evidence once many things were tested.

In [ ]:
# Settings (run.py replaces these)
TICKERS = ["SPY"]

In [ ]:
import sys
from pathlib import Path

# Find the project folder, whether this runs from the project, notebooks/ or reports/
ROOT = Path.cwd()
if ROOT.name in ("notebooks", "reports"):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from IPython.display import Markdown, display

pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams["figure.figsize"] = (11, 4)
# Same colors in every report: blue, orange, aqua, yellow
plt.rcParams["axes.prop_cycle"] = cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100"])

from src.report import comparison_table, load_summaries

summaries = load_summaries(TICKERS)
table, info = comparison_table(summaries)
print(f"Stocks: {', '.join(TICKERS)}")

## 1. Test periods

Each stock is tested from its own first test year, so periods can differ for stocks with shorter histories.

In [ ]:
pd.DataFrame([
    {"stock": s["ticker"], "test start": s["test start"], "test end": s["test end"],
     "test days": s["test days"], "always up accuracy": s["always up accuracy"],
     "buy and hold annual return": s["buy and hold"]["annual return"]}
    for s in summaries
]).set_index("stock")

## 2. Accuracy against "always up"

**Edge** is the model's accuracy minus the "always up" accuracy for the same stock and days. Zero means no better than always predicting up. The bars show the 95% range: if a bar crosses zero, the edge could be luck.

In [ ]:
import numpy as np

plot = table.reset_index()
models = list(dict.fromkeys(plot["model"]))
stocks = list(dict.fromkeys(plot["stock"]))
days = {s["ticker"]: s["test days"] for s in summaries}
colors = {m: c for m, c in zip(models, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"])}

fig, ax = plt.subplots(figsize=(9, 0.6 * len(stocks) * len(models) + 1.5))
y = 0
labels, ticks = [], []
for stock in stocks:
    for model in models:
        row = plot[(plot["stock"] == stock) & (plot["model"] == model)].iloc[0]
        half = 1.96 * np.sqrt(row["accuracy"] * (1 - row["accuracy"]) / days[stock])
        ax.errorbar(row["edge"] * 100, y, xerr=half * 100, fmt="o", color=colors[model],
                    markersize=8, capsize=4, linewidth=2, label=model if stock == stocks[0] else None)
        ticks.append(y)
        labels.append(f"{stock}  {model}")
        y += 1
    y += 0.5
ax.axvline(0, color="grey", linewidth=1)
ax.set_yticks(ticks, labels)
ax.invert_yaxis()
ax.set_xlabel('accuracy minus "always up" (percentage points)')
ax.set_title('Model edge over "always up", with 95% range')
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=len(models), frameon=False)
plt.tight_layout()
plt.show()

## 3. Full table

- **p-value vs always up:** the chance of an accuracy this high if the model were no better than "always up."
- **share of random >=:** from the luck check: how often random cash days did at least as well in the backtest. Empty when a model never left the market.
- The four True/False columns: does the result pass at 5%, and does it pass the corrected bar?

In [ ]:
table

## 4. How many wins would luck produce?

In [ ]:
n = info["comparisons"]
display(Markdown(f'''
- **Comparisons:** {n} ({len(TICKERS)} stocks × {n // max(len(TICKERS), 1)} models)
- **Expected passes by pure luck at 5%:** about {info["expected chance passes"]:.1f} per test
- **Corrected bar:** p-value or luck share below {info["corrected threshold"]:.4f}

| Test | Passes at 5% | Passes after correction |
|---|---|---|
| Accuracy beats "always up" | {info["accuracy passes (5%)"]} of {n} | {info["accuracy passes (corrected)"]} of {n} |
| Backtest beats luck | {info["luck passes (5%)"]} of {n} | {info["luck passes (corrected)"]} of {n} |

A handful of passes at 5%, close to what luck predicts, is **not** evidence of skill. Only results that also pass the corrected bar, and that hold up on future data (phase 6, the live forward test), deserve attention.
'''))

## 5. Conclusions

*Write this in your own words.* Some prompts:

- Did any stock show a real edge after the correction?
- Were results consistent across stocks, or did one stock look special?
- How much did buy-and-hold differ between stocks, and how much of that is hindsight from picking well-known winners?